# General-Purpose Voice Agent

One engine that can run **any** short voice conversation — a booking, a reminder, a
feedback survey, a lead screen. What changes between them is a **checklist**, not code.

The notebook follows the 15 steps of the architecture, one cell per step. Every cell
prints what it produced, so you can check each part before moving on.

```
BEFORE THE CALL      1 describe the goal      2 build the checklist
                     3 check the checklist    4 rehearse      5 pre-record safe lines

EVERY TURN           6 hear start/stop        7 audio -> text
                     8 AI picks the reply     9 code validates the answers
                    10 save with a source    11 safety check, then speak
                    12 (barge-in — needs full-duplex audio, not possible in a notebook)

AFTER THE CALL      13 final record          14 act on it     15 score the call
```

Only `GEMINI_API_KEY` is needed — the same key does the thinking, the listening and the
speaking. Put it in a `.env` next to this notebook, or in Colab Secrets.

In [ ]:
%pip install -q -U "google-genai>=1.0.0" "python-dotenv>=1.0.0" "sounddevice>=0.4.6" "numpy>=1.24"

## A. Setup

In [ ]:
import io, json, logging, os, re, sys, time, wave
from datetime import date, datetime, timedelta

from dotenv import find_dotenv, load_dotenv
from google import genai
from google.genai import errors as genai_errors, types

load_dotenv(find_dotenv(usecwd=True), override=True)

API_KEY = (os.getenv("GEMINI_API_KEY") or "").strip()
if not API_KEY and "google.colab" in sys.modules:
    from google.colab import userdata
    API_KEY = (userdata.get("GEMINI_API_KEY") or "").strip()
if not API_KEY:
    from getpass import getpass
    API_KEY = getpass("GEMINI_API_KEY: ").strip()

client = genai.Client(api_key=API_KEY)

PLAN_MODEL = "gemini-3.5-flash-lite"          # offline steps: compile, rehearse, extract
TURN_MODEL = "gemini-3.5-flash-lite"          # in-call turns; must stay fast
STT_MODEL  = "gemini-3.5-flash-lite"          # audio -> text
TTS_MODEL  = "gemini-3.1-flash-tts-preview"   # text -> audio
AGENT_VOICE, CALLER_VOICE = "Kore", "Puck"

FAST = types.ThinkingConfig(thinking_level="low")
logging.getLogger("google_genai.models").setLevel(logging.ERROR)   # silence SDK advisory

# ── the log ────────────────────────────────────────────────────────────────
# Every step writes exactly one line here. Reading a call is reading this list.
LOG, T0, VERBOSE = [], time.perf_counter(), True

def log_event(stage, **fields):
    e = {"t": round(time.perf_counter() - T0, 2), "stage": stage, **fields}
    LOG.append(e)
    if VERBOSE:
        print(f"  {e['t']:>5.2f}s  {stage:<16} " +
              "  ".join(f"{k}={v}" for k, v in fields.items()))
    return e

def with_retry(call, tries=4):
    """The free tier allows about 15 requests a minute, and a rehearsal outruns that.
    Wait for the server's own retry delay rather than failing the run."""
    for attempt in range(tries):
        try:
            return call()
        except genai_errors.ClientError as e:
            if getattr(e, "code", None) != 429 or attempt == tries - 1:
                raise
            m = re.search(r"retryDelay[\"']?:\s*[\"']?(\d+)", str(e))
            wait = min(int(m.group(1)) + 2 if m else 20 * (attempt + 1), 65)
            print(f"  rate limited — waiting {wait}s")
            time.sleep(wait)

def ask_json(model, prompt, schema, max_tokens=800, system=None, temperature=0.3):
    """One model call that is guaranteed to come back as schema-valid JSON."""
    resp = with_retry(lambda: client.models.generate_content(
        model=model, contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system, temperature=temperature,
            max_output_tokens=max_tokens, response_mime_type="application/json",
            response_schema=schema, thinking_config=FAST)))
    text = (resp.text or "").strip()
    if not text:
        # Almost always max_tokens: the model was cut off before it closed the JSON.
        finish = resp.candidates[0].finish_reason if resp.candidates else "?"
        raise RuntimeError(f"empty reply from {model} (finish_reason={finish}); raise max_tokens")
    return json.loads(text)

print("Ready. Models:", TURN_MODEL, "|", TTS_MODEL)

### A2. Audio helpers

Three small functions the rest of the notebook builds on: **speak**, **record**, **transcribe**.
If there is no sound hardware (Colab, a server), everything still runs — it just goes quiet,
and the call is saved to a WAV you can play at the end.

In [ ]:
try:
    import numpy as np, sounddevice as sd
    AUDIO = True
except Exception as e:
    AUDIO, _audio_err = False, e

TTS_RATE, MIC_RATE = 24000, 16000
LOOPBACK = ("stereo mix", "what u hear", "loopback", "line in")

HOST_ORDER = ["wasapi", "directsound", "mme", "wdm-ks"]   # last one often refuses to open

def mic_opens(device):
    """A device can be listed and still fail with 'Invalid device' when you open it —
    common for WDM-KS entries on Windows. The only reliable test is to try."""
    try:
        with sd.InputStream(samplerate=MIC_RATE, channels=1, dtype="int16",
                            blocksize=480, device=device):
            return True
    except Exception:
        return False

def pick_device(kind):
    """Pick a device index that actually works, or None.

    Two traps: Windows often has no *default* recording device, and loopback inputs record
    the speakers, which would make the agent transcribe itself."""
    if not AUDIO:
        return None
    key = "max_input_channels" if kind == "in" else "max_output_channels"
    hosts = {i: a["name"].lower() for i, a in enumerate(sd.query_hostapis())}
    found = [(i, d) for i, d in enumerate(sd.query_devices()) if d[key] > 0]
    if kind == "in":
        found = [(i, d) for i, d in found
                 if not any(h in d["name"].lower() for h in LOOPBACK)]
    found.sort(key=lambda x: next((n for n, h in enumerate(HOST_ORDER)
                                   if h in hosts.get(x[1]["hostapi"], "")), 9))
    if kind == "out":
        return found[0][0] if found else None
    return next((i for i, _ in found if mic_opens(i)), None)

MIC, SPEAKER = pick_device("in"), pick_device("out")
RECORDING = bytearray()          # everything anyone said, for playback at the end

def pcm_to_wav(pcm, rate):
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate); w.writeframes(pcm)
    return buf.getvalue()

def play(pcm):
    """Play 24 kHz PCM and keep a copy for the call recording."""
    RECORDING.extend(pcm)
    if not (AUDIO and SPEAKER is not None and pcm):
        return
    sd.play(np.frombuffer(pcm, dtype=np.int16), TTS_RATE, device=SPEAKER)
    sd.wait()

def speak_pcm(text, language, voice=AGENT_VOICE):
    """Text -> audio bytes. Streamed, so the first chunk arrives in ~1.5s."""
    cfg = types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(voice_config=types.VoiceConfig(
            prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=voice))))
    out = bytearray()
    for chunk in client.models.generate_content_stream(
            model=TTS_MODEL, config=cfg,
            contents=f"Say this in {language}, warm and unhurried. Only these words:\n{text}"):
        try:
            out.extend(chunk.candidates[0].content.parts[0].inline_data.data or b"")
        except (AttributeError, IndexError, TypeError):
            pass
    return bytes(out)

def transcribe(wav_bytes, language):
    """Audio -> text. Returns "" when nobody spoke."""
    resp = with_retry(lambda: client.models.generate_content(
        model=STT_MODEL,
        contents=[types.Part.from_bytes(data=wav_bytes, mime_type="audio/wav"),
                  f"Transcribe this phone-call audio verbatim in {language}. "
                  f"Return only the words. If there is no speech, return: <none>"],
        config=types.GenerateContentConfig(temperature=0, max_output_tokens=256,
                                           thinking_config=FAST)))
    text = (resp.text or "").strip().strip('"')
    return "" if text.lower() in ("<none>", "none", "") else text

print("audio:", "on" if AUDIO else f"off ({_audio_err})",
      "| mic:", MIC if MIC is not None else "none that opens",
      "| speaker:", SPEAKER)
if AUDIO and MIC is None:
    print("  No usable microphone — the notebook runs with a fake customer instead.")

## Step 1 — Describe the goal

Plain English. This is the only thing a business user ever writes.

In [ ]:
GOAL = ("Call customers who booked a table to confirm the booking: "
        "which day, what time, how many people, and the name it is under.")
LANGUAGE = "English"

print(GOAL)

## Step 2 — Build the checklist

The goal becomes a **checklist** of things the conversation must collect. Each item says
what to ask and what a valid answer looks like — that second part is what lets ordinary
code check the answers later, in Step 9.

Swap the `GOAL` above for *"remind patients about tomorrow's appointment"* and re-run:
the checklist changes, and nothing else in this notebook does.

In [ ]:
SLOT_SCHEMA = types.Schema(type="OBJECT", properties={
    "id":      types.Schema(type="STRING", description="short snake_case name"),
    "ask":     types.Schema(type="STRING", description="the question, spoken aloud, under 15 words"),
    "type":    types.Schema(type="STRING", enum=["number", "choice", "text", "time", "date"]),
    "options": types.Schema(type="ARRAY", nullable=True, items=types.Schema(type="STRING"),
                            description="allowed answers, only when type is choice"),
    "min":     types.Schema(type="NUMBER", nullable=True),
    "max":     types.Schema(type="NUMBER", nullable=True),
}, required=["id", "ask", "type"])

SPEC_SCHEMA = types.Schema(type="OBJECT", properties={
    "goal":     types.Schema(type="STRING"),
    "slots":    types.Schema(type="ARRAY", max_items=5, items=SLOT_SCHEMA),
    "goodbye":  types.Schema(type="STRING", description="warm sign-off, under 12 words"),
    "no_info":  types.Schema(type="STRING", description="said when the customer asks about something the agent was never told, e.g. prices or opening hours; must offer a colleague follow-up"),
    "wrap_up":  types.Schema(type="STRING", description="said when time runs out"),
}, required=["goal", "slots", "goodbye", "no_info", "wrap_up"])

COMPILE_PROMPT = """Design a short phone call for this goal:
{goal}

The call is in {language} and must finish in under 90 seconds, so list at most 4 things to
collect. For each one:
- "ask" is one sentence a person says out loud: under 15 words, no lists, digits as digits.
- "type" decides how the answer is checked: number (give min and max), choice (give options),
  time, date, or text for anything open-ended.
The three closing lines are spoken aloud too, so keep them short and natural."""

def compile_spec(goal, language):
    spec = ask_json(PLAN_MODEL, COMPILE_PROMPT.format(goal=goal, language=language),
                    SPEC_SCHEMA, max_tokens=1200, temperature=0.4)
    spec["language"] = language
    spec["budget_s"] = 90
    return spec

spec = compile_spec(GOAL, LANGUAGE)

print(f"{spec['goal']}\n")
for s in spec["slots"]:
    limits = (f"options={s['options']}" if s.get("options") else
              f"{s.get('min')}..{s.get('max')}" if s.get("min") is not None else "")
    print(f"  {s['id']:<14} {s['type']:<7} {limits:<22} \"{s['ask']}\"")
print(f"\n  goodbye: {spec['goodbye']}\n  no_info: {spec['no_info']}\n  wrap_up: {spec['wrap_up']}")

## Step 3 — Check the checklist

A bad checklist should fail here, not on a live call. These are plain rules, no AI:
a question too long to say, a choice with no options, a number with no range, a duplicate name.

In [ ]:
def lint(spec):
    problems, seen = [], set()
    if not spec["slots"]:
        problems.append("no slots — the call would have nothing to ask")
    for s in spec["slots"]:
        if s["id"] in seen:
            problems.append(f"{s['id']}: duplicate name")
        seen.add(s["id"])
        if len(s["ask"].split()) > 18:
            problems.append(f"{s['id']}: question is {len(s['ask'].split())} words, too long to say")
        if s["type"] == "choice" and not s.get("options"):
            problems.append(f"{s['id']}: type is choice but no options given")
        if s["type"] == "number" and (s.get("min") is None or s.get("max") is None):
            problems.append(f"{s['id']}: type is number but no range given")
    for line in ("goodbye", "no_info", "wrap_up"):
        if not spec.get(line, "").strip():
            problems.append(f"missing the {line} line")
    return problems

problems = lint(spec)
print("\n".join(f"  x {p}" for p in problems) if problems else "  ok — checklist is usable")

## Step 5 — Pre-record the safe lines

The goodbye, the *"I don't have that"* and the wrap-up are recorded **now**, once.

This is what stops the safety check in Step 11 from costing anything. When a sentence has to
be replaced mid-call, the replacement audio already exists — no new model call, no new
synthesis, no two-second gap.

In [ ]:
def record_lines(spec):
    return {name: speak_pcm(spec[name], spec["language"]) for name in ("goodbye", "no_info", "wrap_up")}

LINES = record_lines(spec)
for name, pcm in LINES.items():
    print(f"  {name:<9} {len(pcm) / 2 / TTS_RATE:>4.1f}s   \"{spec[name]}\"")

## Steps 6 & 7 — Ears: hear the customer, turn it into text

Three interchangeable versions, same one method. `listen()` is given what the agent just said,
which only the fake caller uses.

- **MicEars** — a real microphone. Waits for you to start talking, stops after a second of quiet.
- **FakeEars** — an LLM plays the customer. No audio, so it is fast and free: this is what Step 4 rehearses with.
- **TypedEars** — you type. The fallback when there is no sound hardware.

In [ ]:
SILENCE_TAIL, MAX_UTTERANCE, NOISE_FLOOR = 1.0, 12.0, 0.004

class MicEars:
    """Records until you stop speaking, then transcribes."""
    def __init__(self, language, device=MIC):
        self.language, self.device = language, device

    def listen(self, agent_said=""):
        pcm = self._record()
        if len(pcm) / 2 / MIC_RATE < 0.35:
            log_event("ears", heard="<silence>")
            return ""
        RECORDING.extend(self._to_24k(pcm))
        t0 = time.perf_counter()
        text = transcribe(pcm_to_wav(pcm, MIC_RATE), self.language)
        log_event("ears", heard=repr(text), ms=int((time.perf_counter() - t0) * 1000))
        return text

    def _record(self, wait_s=8.0):
        import queue
        q, frames, started = queue.Queue(), [], False
        t0 = last = time.perf_counter()
        with sd.InputStream(samplerate=MIC_RATE, channels=1, dtype="int16",
                            blocksize=480, device=self.device,
                            callback=lambda d, *_: q.put(bytes(d))):
            while True:
                try:
                    buf = q.get(timeout=0.15)
                except queue.Empty:
                    buf = None
                now = time.perf_counter()
                if buf is not None:
                    loud = np.abs(np.frombuffer(buf, dtype=np.int16)).mean() / 32768 > NOISE_FLOOR * 4
                    if loud:
                        started, last = True, now
                    if started:
                        frames.append(buf)
                    else:
                        frames = (frames + [buf])[-8:]      # pre-roll, so no clipped first word
                if not started and now - t0 > wait_s:
                    return b""
                if started and (now - last > SILENCE_TAIL or now - t0 > MAX_UTTERANCE):
                    return b"".join(frames)

    @staticmethod
    def _to_24k(pcm):
        src = np.frombuffer(pcm, dtype=np.int16).astype(np.float32)
        n = int(len(src) * TTS_RATE / MIC_RATE)
        return np.interp(np.linspace(0, len(src) - 1, n), np.arange(len(src)),
                         src).astype(np.int16).tobytes()


class FakeEars:
    """An LLM plays the customer. Used for rehearsal — no microphone, no audio cost."""
    def __init__(self, persona, spec):
        self.persona, self.spec, self.history = persona, spec, []

    def listen(self, agent_said=""):
        # Without the history a fake caller repeats itself forever and no call ever ends.
        recent = "\n".join(self.history[-6:]) or "(the call just started)"
        self.history.append(f"agent: {agent_said}")
        reply = ask_json(PLAN_MODEL,
            f"You are a customer on a phone call. {self.persona}\n"
            f"The call is about: {self.spec['goal']}\n"
            f"So far:\n{recent}\n"
            f"The agent just said: \"{agent_said}\"\n"
            f"Reply in one short spoken sentence. Never ask something you already asked — "
            f"if they could not answer it, let it go and answer their question instead.",
            types.Schema(type="OBJECT", properties={"reply": types.Schema(type="STRING")},
                         required=["reply"]), max_tokens=300, temperature=0.9)["reply"]
        self.history.append(f"customer: {reply}")
        log_event("ears", heard=repr(reply))
        return reply


class TypedEars:
    def listen(self, agent_said=""):
        text = input("  you: ").strip()
        log_event("ears", heard=repr(text))
        return text

print("MicEars / FakeEars / TypedEars defined.  mic available:", MIC is not None)

## Step 11a — Mouth: say it out loud

`say()` synthesises a fresh sentence. `say_line()` plays one of the recordings from Step 5 —
instant, because the audio already exists.

In [ ]:
class Mouth:
    def __init__(self, spec, lines):
        self.spec, self.lines = spec, lines

    def say(self, text):
        t0 = time.perf_counter()
        play(speak_pcm(text, self.spec["language"]))
        log_event("mouth", said=repr(text), ms=int((time.perf_counter() - t0) * 1000))

    def say_line(self, name):
        play(self.lines[name])
        log_event("mouth", said=repr(self.spec[name]), source="pre-recorded", ms=0)
        return self.spec[name]

mouth = Mouth(spec, LINES)
mouth.say_line("no_info")          # check: you should hear this, instantly

## Step 9 — Validate the answers (plain code, no AI)

The model *proposes* an answer; this decides whether it counts. Each checker returns either
`(True, cleaned_value)` or `(False, reason)` — and the reason is logged, which is how you find
out why an answer never made it into the record.

`"seven"` for a time is rejected on purpose: seven in the morning and seven in the evening are
different bookings, so the agent has to ask again.

In [ ]:
WORDS = {"one":1,"two":2,"three":3,"four":4,"five":5,"six":6,"seven":7,"eight":8,"nine":9,
         "ten":10,"eleven":11,"twelve":12,"fifteen":15,"twenty":20,"thirty":30,"fifty":50,
         "a couple":2,"couple":2,"a few":3}
WEEKDAYS = ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"]

def _num(raw):
    raw = str(raw).strip().lower()
    if raw.replace(".", "", 1).isdigit():
        return float(raw)
    for word, n in WORDS.items():
        if re.search(rf"\b{word}\b", raw):
            return float(n)
    return None

def check(slot, raw):
    """(ok, value) or (False, reason). One branch per type — nothing clever."""
    raw = str(raw).strip()
    if not raw:
        return False, "empty"

    if slot["type"] == "number":
        n = _num(raw)
        if n is None:
            return False, "not_a_number"
        lo, hi = slot.get("min", 0), slot.get("max", 1e9)
        return (True, int(n)) if lo <= n <= hi else (False, f"outside_{int(lo)}_{int(hi)}")

    if slot["type"] == "choice":
        for option in slot.get("options", []):
            if re.search(rf"\b{re.escape(option.lower())}\b", raw.lower()):
                return True, option
        return False, "not_one_of_the_options"

    if slot["type"] == "time":
        m = re.search(r"(\d{1,2})[:.]?(\d{2})?\s*(am|pm)?", raw.lower())
        hour = int(m.group(1)) if m else _num(raw)
        if hour is None:
            return False, "no_time_found"
        hour, minute, ampm = int(hour), int(m.group(2) or 0) if m else 0, (m.group(3) if m else None)
        if hour > 23 or minute > 59:
            return False, "impossible_time"
        if ampm == "pm" and hour < 12:
            hour += 12
        elif ampm == "am" and hour == 12:
            hour = 0
        elif ampm is None and hour < 12:
            return False, "ambiguous_am_pm"      # 7 could be breakfast or dinner
        return True, f"{hour:02d}:{minute:02d}"

    if slot["type"] == "date":
        low, today = raw.lower(), date.today()
        if "today" in low:
            return True, str(today)
        if "day after" in low:
            return True, str(today + timedelta(days=2))
        if "tomorrow" in low:
            return True, str(today + timedelta(days=1))
        for i, day in enumerate(WEEKDAYS):
            if day in low:
                return True, str(today + timedelta(days=(i - today.weekday()) % 7 or 7))
        m = re.search(r"(\d{1,2})[/-](\d{1,2})", low)
        if m:
            return True, f"{today.year}-{int(m.group(2)):02d}-{int(m.group(1)):02d}"
        return False, "no_date_found"

    return (True, raw) if len(raw) > 1 else (False, "too_short")


# check: three that must pass, three that must fail
_time = {"id": "time", "type": "time"}
for slot, raw in [({"id":"n","type":"number","min":1,"max":12}, "four"), (_time, "7pm"),
                  ({"id":"d","type":"date"}, "tomorrow"),
                  ({"id":"n","type":"number","min":1,"max":12}, "fifty"), (_time, "seven"),
                  ({"id":"c","type":"choice","options":["yes","no"]}, "maybe")]:
    ok, out = check(slot, raw)
    print(f"  {'ok  ' if ok else 'no  '} {slot['type']:<7} {raw!r:<12} -> {out}")

## Step 8 — The AI picks the next reply

One model call per turn. It is given the checklist, what is filled in so far, and what the
customer just said. It returns the sentence to say plus the answers it *thinks* it just heard —
proposals only; Step 9 decides whether they count.

The function is **stateless**: same inputs, same output, no hidden chat history. That means you
can replay any turn from the log later without re-running the call.

In [ ]:
TURN_SCHEMA = types.Schema(type="OBJECT", properties={
    "say":     types.Schema(type="STRING", description="what to say next, under 25 spoken words"),
    "answers": types.Schema(type="ARRAY", max_items=4, items=types.Schema(type="OBJECT", properties={
                   "slot":  types.Schema(type="STRING"),
                   "value": types.Schema(type="STRING")}, required=["slot", "value"])),
    "done":    types.Schema(type="BOOLEAN", description="true only when saying goodbye"),
}, required=["say", "answers", "done"])

TURN_SYSTEM = """You are making a short phone call for this purpose: {goal}
Speak only {language}. Warm, brief, natural — at most 2 sentences and 25 words per turn.

Still to find out:
{todo}
Already answered:
{done}

Rules:
- Ask about one missing item at a time, using its question as a guide. Never re-ask something answered.
- "answers" is only for what the customer actually said this turn. Never guess, never fill it from
  your own question. If they said nothing usable, leave it empty.
- If they ask you something you were not told above (prices, hours, someone else's booking), say you
  do not have that detail and a colleague will follow up — then ask your question again.
- Never promise a refund, a discount, a callback time, or anything else.
- Say digits as digits. No lists, no markdown, nothing they would have to look at.
- Read dates and times back the way a person says them — "Friday the eighteenth", "seven in the
  evening" — never "2026-09-18" or "19:00", even though that is how they are stored.
- Set "done" true only when the sentence you are returning is the goodbye."""

def next_turn(spec, state, heard, seconds_left, stuck=0):
    todo = "\n".join(f"- {s['id']}: {s['ask']}" for s in spec["slots"] if s["id"] not in state) or "- nothing"
    done = "\n".join(f"- {k} = {v['value']}" for k, v in state.items()) or "- nothing yet"
    prompt = (f"The customer just said: \"{heard}\"" if heard else
              "The call has just connected. Greet them in one clause and ask your first question.")
    if seconds_left < 20:
        prompt += f"\n(Only {int(seconds_left)} seconds left — start wrapping up.)"
    t0 = time.perf_counter()
    turn = ask_json(TURN_MODEL, prompt, TURN_SCHEMA, max_tokens=220, temperature=0.4,
                    system=TURN_SYSTEM.format(goal=spec["goal"], language=spec["language"],
                                              todo=todo, done=done))
    log_event("brain", proposed=[f"{a['slot']}={a['value']}" for a in turn["answers"]],
              ms=int((time.perf_counter() - t0) * 1000))
    return turn

demo = next_turn(spec, {}, "", 90)
print("\n  says:", demo["say"], "\n  proposes:", demo["answers"], "\n  done:", demo["done"])

## Step 11b — Safety check

Word matching only, no AI, so it takes well under a millisecond and adds no waiting time.

Four outcomes, never just yes/no — and **none of them asks the model to write a new sentence**,
which is what used to cost two seconds:

| outcome | what happens | cost |
|---|---|---|
| `speak` | say it as written | 0 |
| `trim`  | delete the offending words, say the rest | 0 |
| `swap`  | play a pre-recorded line from Step 5 | 0 |
| `wrap`  | play the wrap-up line and end the call | 0 |

Each rule has a **mode**. A new rule starts as `watch`: it writes a log line and the call carries
on as normal. Only once you have looked at those logs and seen it is reliably right do you switch
it to `enforce`. That way a badly written rule can waste log space, but it can never break a call.

In [ ]:
RULES = [
    {"id": "promise",   "mode": "enforce", "action": "swap", "line": "no_info",
     "pattern": r"\b(guarantee|guaranteed|definitely|promise|refund|discount|free of charge)\b"},
    {"id": "long_digits", "mode": "enforce", "action": "trim",
     "pattern": r"\b\d{6,}\b"},                      # never read a card or phone number aloud
    {"id": "hedging",   "mode": "watch",   "action": "swap", "line": "no_info",
     "pattern": r"\b(i think|probably|not sure|maybe)\b"},
]

def screen(text):
    """-> (action, payload, rule_id). action is speak | trim | swap | wrap."""
    for rule in RULES:
        if not re.search(rule["pattern"], text, re.I):
            continue
        if rule["mode"] == "watch":
            log_event("guard", rule=rule["id"], mode="watch", spoken_anyway=True)
            continue
        if rule["action"] == "trim":
            cleaned = re.sub(rule["pattern"], "that number", text, flags=re.I)
            log_event("guard", rule=rule["id"], action="trim")
            return "speak", cleaned, rule["id"]
        log_event("guard", rule=rule["id"], action=rule["action"])
        return rule["action"], rule.get("line"), rule["id"]
    return "speak", text, None

for sample in ["Great, see you Friday at seven.",
               "I can definitely get you a refund for that.",
               "Your booking reference is 884413320.",
               "I think it was probably fine."]:
    action, payload, rule = screen(sample)
    print(f"  {action:<6} {str(rule):<12} {payload}")

## The call — steps 6 to 12 in a loop

Read top to bottom: it is the seven steps in order, and nothing else. The loop, not the model,
decides when the call ends — "the bot won't hang up" is then always a bug in one place.

Step 12 (stopping when the customer interrupts) needs the microphone open while the speaker is
playing. A notebook cell cannot do that, so it is marked and left out.

In [ ]:
def run_call(spec, ears, mouth, max_turns=8):
    global T0, LOG
    T0, LOG = time.perf_counter(), []
    RECORDING.clear()
    state, heard, stuck = {}, "", 0
    log_event("call.start", goal=spec["goal"][:40], slots=len(spec["slots"]))

    for turn in range(1, max_turns + 1):
        left = spec["budget_s"] - (time.perf_counter() - T0)
        if left < 3:
            mouth.say_line("wrap_up")
            log_event("call.end", reason="out_of_time", filled=len(state))
            return state

        reply = next_turn(spec, state, heard, left, stuck)              # step 8

        before = len(state)
        for a in reply["answers"]:                                      # step 9
            slot = next((s for s in spec["slots"] if s["id"] == a["slot"]), None)
            if slot is None:
                log_event("reject", slot=a["slot"], reason="not_on_checklist")
                continue
            ok, value = check(slot, a["value"])
            if ok:
                state[slot["id"]] = {"value": value, "from_turn": turn}  # step 10
                log_event("accept", slot=slot["id"], value=value, from_turn=turn)
            else:
                log_event("reject", slot=slot["id"], raw=a["value"], reason=value)

        # Nobody is moving: the customer keeps dodging, or the model keeps missing the answer.
        # Three turns of that is a stuck call, and code ends it rather than hoping.
        stuck = 0 if len(state) > before else stuck + 1
        if stuck >= 3:
            mouth.say_line("wrap_up")
            log_event("call.end", reason="no_progress", filled=len(state))
            return state

        action, payload, _ = screen(reply["say"])                       # step 11
        if action == "speak":
            mouth.say(payload)
        elif action == "swap":
            mouth.say_line(payload)
        else:
            mouth.say_line("wrap_up")
            log_event("call.end", reason="guard_wrap", filled=len(state))
            return state

        if len(state) == len(spec["slots"]):                            # everything collected
            if not reply["done"]:      # it already signed off — don't say goodbye twice
                mouth.say_line("goodbye")
            log_event("call.end", reason="complete", filled=len(state))
            return state
        if reply["done"]:
            log_event("call.end", reason="agent_ended", filled=len(state))
            return state

        heard = ears.listen(reply["say"])                               # steps 6 + 7
        # step 12 (barge-in) would go here: cancel playback the moment speech is detected.

    log_event("call.end", reason="max_turns", filled=len(state))
    return state

print("run_call defined — steps 8, 9, 10, 11, 6, 7 in that order, nothing hidden")

## Step 4 — Rehearse with fake customers

Four awkward callers, no audio, so it is quick and costs almost nothing. A checklist that cannot
survive these should not be taking real calls.

In [ ]:
PERSONAS = {
    "helpful":  "You answer every question directly and briefly.",
    "rambler":  "You wander off topic, ask what time they close, and take a while to answer.",
    "suspicious": "You first demand to know who is calling and why, then cooperate.",
    "terse":    "You answer in one or two words and volunteer nothing.",
}

class SilentMouth(Mouth):
    """Same interface, no audio — rehearsal should not wait on speech synthesis."""
    def say(self, text):
        log_event("mouth", said=repr(text))
    def say_line(self, name):
        log_event("mouth", said=repr(self.spec[name]), source="pre-recorded")
        return self.spec[name]

def rehearse(spec, personas=PERSONAS):
    global VERBOSE
    VERBOSE, results = False, []
    for name, persona in personas.items():
        state = run_call(spec, FakeEars(persona, spec), SilentMouth(spec, LINES))
        results.append((name, state, [e for e in LOG if e["stage"] == "reject"],
                        next(e for e in LOG if e["stage"] == "call.end")["reason"]))
    VERBOSE = True

    need = len(spec["slots"])
    for name, state, rejects, reason in results:
        mark = "pass" if len(state) == need else "FAIL"
        print(f"  {mark}  {name:<11} {len(state)}/{need} filled  ended={reason:<12} "
              f"rejects={len(rejects)}")
    return results

_ = rehearse(spec)

## Run the call

`MicEars` if you have a microphone — talk to it. Otherwise a fake customer plays the other side,
out loud, so you can hear the whole thing.

In [ ]:
ears = MicEars(spec["language"]) if MIC is not None else FakeEars(PERSONAS["helpful"], spec)
print("customer side:", type(ears).__name__, "\n")

state = run_call(spec, ears, mouth)

print("\ncollected:")
for k, v in state.items():
    print(f"  {k:<14} {v['value']}   (from turn {v['from_turn']})")

## Reading the trace

One line per step. To debug a call you walk this top-down and stop at the first line that
disagrees with what actually happened — that stage is the culprit.

In [ ]:
def show_trace(log):
    for e in log:
        rest = "  ".join(f"{k}={v}" for k, v in e.items() if k not in ("t", "stage"))
        print(f"{e['t']:>6.2f}s  {e['stage']:<12} {rest}")

show_trace(LOG)

## Step 13 — The final record

The in-call model was rushed and saw one turn at a time. This pass reads the whole conversation
at once with no time pressure — then the two are **compared**. Where they disagree, the call is
flagged for a human. That flag is the cheapest quality signal in the whole system.

In [ ]:
def final_record(spec, log, live_state):
    transcript = "\n".join(
        f"agent: {e['said']}" if e["stage"] == "mouth" else f"customer: {e['heard']}"
        for e in log if e["stage"] in ("mouth", "ears"))

    schema = types.Schema(type="OBJECT", properties={
        "answers": types.Schema(type="ARRAY", items=types.Schema(type="OBJECT", properties={
            "slot": types.Schema(type="STRING"), "value": types.Schema(type="STRING")},
            required=["slot", "value"])),
        "summary": types.Schema(type="STRING", description="one factual sentence"),
        "follow_up": types.Schema(type="BOOLEAN", description="true if the customer was unhappy or asked for a human"),
    }, required=["answers", "summary", "follow_up"])

    found = ask_json(PLAN_MODEL,
        f"Read this phone call and report only what the customer actually said.\n"
        f"Things the call was collecting: {[s['id'] for s in spec['slots']]}\n"
        f"Never infer a value they did not give — leave it out instead.\n\n{transcript}",
        schema, max_tokens=700)

    record, flags = {}, []
    for a in found["answers"]:
        slot = next((s for s in spec["slots"] if s["id"] == a["slot"]), None)
        if not slot:
            continue
        ok, value = check(slot, a["value"])
        if ok:
            record[slot["id"]] = value
    for slot_id in set(record) | set(live_state):
        live = live_state.get(slot_id, {}).get("value")
        if str(record.get(slot_id)) != str(live):
            flags.append(f"{slot_id}: live={live} vs transcript={record.get(slot_id)}")

    return {"answers": record, "summary": found["summary"],
            "follow_up": found["follow_up"], "flags": flags}

record = final_record(spec, LOG, state)
print(json.dumps(record, indent=2))
print("\n  " + ("flagged for review" if record["flags"] else "live and transcript agree"))

## Steps 14 & 15 — Act on it, then score the call

`act()` is a stub that prints what it would do — this is where a real system writes to the
booking system, the calendar or the CRM. `score()` is what tells you whether a prompt change
made things better or worse, instead of guessing.

In [ ]:
def act(spec, record):
    if not record["answers"]:
        return "nothing to act on"
    fields = ", ".join(f"{k}={v}" for k, v in record["answers"].items())
    return f"would submit -> {fields}" + ("   [+ human follow-up]" if record["follow_up"] else "")

def score(spec, log, state, record):
    end = next(e for e in log if e["stage"] == "call.end")
    return {
        "goal_met":    len(state) == len(spec["slots"]),
        "filled":      f"{len(state)}/{len(spec['slots'])}",
        "seconds":     log[-1]["t"],
        "within_budget": log[-1]["t"] <= spec["budget_s"],
        "turns":       sum(1 for e in log if e["stage"] == "brain"),
        "rejected":    [f"{e.get('slot')}:{e.get('reason')}" for e in log if e["stage"] == "reject"],
        "guard_hits":  [e["rule"] for e in log if e["stage"] == "guard"],
        "ended":       end["reason"],
        "flagged":     bool(record["flags"]),
    }

print(act(spec, record), "\n")
for k, v in score(spec, LOG, state, record).items():
    print(f"  {k:<15} {v}")

## Listen to the call

Both sides, as they were spoken. This is also how you hear the call on Colab, where there is no
local sound device.

In [ ]:
from IPython.display import Audio, display

wav = pcm_to_wav(bytes(RECORDING), TTS_RATE)
with open("call.wav", "wb") as f:
    f.write(wav)
print(f"{len(RECORDING) / 2 / TTS_RATE:.1f}s saved to call.wav")
display(Audio(wav, rate=TTS_RATE))

## Try a different use case

Nothing below the checklist knows what a booking is. Change the goal, re-run Steps 2, 3, 5 and
the call — the same seven in-call steps handle it.

```python
GOAL = "Remind patients about tomorrow's appointment and find out if they are coming."
GOAL = "Screen loan enquiries: how much they want, their monthly income, and their city."
GOAL = "Ask customers who just ate how the food and the service were."
```

In [ ]:
GOAL2 = "Remind patients about tomorrow's appointment and find out whether they are coming."
spec2 = compile_spec(GOAL2, "English")

print(lint(spec2) or "  checklist ok")
for s in spec2["slots"]:
    print(f"  {s['id']:<14} {s['type']:<7} \"{s['ask']}\"")

# Same engine, different checklist:
# LINES2 = record_lines(spec2)
# state2 = run_call(spec2, FakeEars(PERSONAS["terse"], spec2), Mouth(spec2, LINES2))